# NLP Mastery Journey — Module 5: Classical Machine Learning for NLP

You can now turn text into numbers (Module 3) and into meaning-aware vectors (Module 4). This module is where it starts producing **predictions**: sentiment labels, spam/not-spam, topic categories. These "classical" models are not a historical footnote — **Naive Bayes, Logistic Regression, linear SVMs, and gradient-boosted trees are still running in production at every major tech company today**, for one very practical reason spelled out below.

### Why big tech still ships these models (not just LLMs)
A large language model call costs milliseconds-to-seconds and real money, *per request*. A linear model scores a request in **microseconds for effectively free**. So the actual industry pattern — at Google, Meta, Amazon, and yes, at Anthropic itself — is: **cheap classical models handle the massive, latency-critical, high-volume traffic** (spam filters, first-pass content moderation, ranking signals, intent routing), and **LLMs get reserved for the harder minority of cases** that classical models flag as uncertain. Knowing how to build the classical layer well is a genuinely production-relevant skill, not a stepping stone to forget.

### What this notebook teaches
| # | Topic | Industry relevance |
|---|-------|----------------------|
| 1 | Naive Bayes | The classic spam-filter algorithm; still a strong, tiny, fast baseline |
| 2 | Logistic Regression | The single most common production text classifier — fast, interpretable, calibratable |
| 3 | Linear SVMs | Excellent on high-dimensional sparse text features (TF-IDF) |
| 4 | Gradient-boosted trees | Standard when combining text features with structured/tabular signals |
| 5 | Cross-validation & hyperparameter tuning | How you actually pick a model + settings responsibly |
| 6 | Evaluation deep-dive | Precision/recall/F1, confusion matrices, ROC-AUC, PR-AUC |
| 7 | Handling class imbalance | Real-world data is rarely balanced (fraud, abuse, spam...) |
| 8 | Interpretability | Explaining *why* a model predicted what it predicted |
| 9 | Production: calibration, thresholds, monitoring | Turning a notebook model into a safe production system |

### How to use this notebook
- Every code cell here **runs live**, top to bottom, on a small synthetic dataset built right in the notebook — you'll see real numbers, real confusion matrices, real plots, not just descriptions.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.
- Comments explain *why* each line exists, not just what it does — read them on your first pass.


## 0. Setup

In [ ]:
# %pip install scikit-learn pandas numpy matplotlib
# Industry-standard extras you'll likely add on a real project (commented
# out here since they need installing, but the patterns below show their use):
# %pip install xgboost lightgbm imbalanced-learn shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42   # reused everywhere below so every result in this
                     # notebook is exactly reproducible when you re-run it
print("Setup note: uncomment the pip install lines above the first time you run this.")


## Part 1 — A Small, Realistic Dataset to Learn On

We build a synthetic sentiment dataset (positive/negative movie-style reviews) right here, so every cell in this notebook is immediately runnable — no downloads, no waiting. It combines a handful of natural, hand-written reviews with a larger batch generated from templates (a real, commonly-used technique for quickly bootstrapping a labeled dataset), plus a touch of injected label noise so the results you see below look like a **realistic** dataset — genuinely learnable, but not suspiciously perfect. The patterns are 1:1 identical to what you'd do with a real CSV loaded via `pd.read_csv()` (Module 1).


In [ ]:
import random
random.seed(RANDOM_STATE)

# ── A few natural, hand-written reviews for lexical variety ──────────────────
hand_written_positive = [
    "this movie was absolutely fantastic and I loved every minute of it",
    "brilliant performances and a gripping story from start to finish",
    "one of the best films I have seen this year, truly excellent",
    "the acting was superb and the plot kept me hooked throughout",
    "a wonderful, heartwarming movie that I would happily watch again",
    "amazing cinematography and a genuinely touching story",
    "the cast delivered outstanding performances, highly recommended",
    "a masterpiece with brilliant direction and an emotional payoff",
    "great movie, funny, smart, and beautifully shot",
    "I really enjoyed this film, the writing was clever and sharp",
]
hand_written_negative = [
    "this movie was boring and a complete waste of time",
    "terrible acting and a plot that made no sense at all",
    "one of the worst films I have ever sat through, awful",
    "the story was dull and the pacing dragged on forever",
    "a disappointing movie with flat, lifeless performances",
    "poor writing and an ending that felt completely unearned",
    "the cast seemed disengaged, the whole thing felt lazy",
    "a mess of a film with no coherent plot or direction",
    "bad movie, unfunny, predictable, and badly shot",
    "I did not enjoy this film, the dialogue was clunky and forced",
]

# ── Template-based generation: a real, fast way to bootstrap labeled data ───
# Real teams do exactly this (mixed with real user data) when they don't yet
# have enough labeled examples for a new category.
positive_adjectives = ["fantastic", "brilliant", "excellent", "superb", "wonderful", "amazing",
                        "outstanding", "incredible", "delightful", "captivating", "charming",
                        "masterful", "exceptional", "gripping", "touching"]
negative_adjectives = ["terrible", "boring", "awful", "dull", "disappointing", "lifeless",
                        "poor", "clunky", "confusing", "forgettable", "tedious", "weak",
                        "flat", "hollow", "depressing"]
templates = [
    "this movie was {adj} and I {feel} every minute of it",
    "the acting was {adj} and the story was {adj2}",
    "a {adj} film with {adj2} performances from the whole cast",
    "the plot was {adj}, and the pacing felt {adj2}",
    "I found the movie to be {adj}, with {adj2} writing throughout",
    "the director delivered a {adj} experience with {adj2} visuals",
    "overall this was a {adj} film, {adj2} from start to finish",
    "the soundtrack was {adj} and the ending felt {adj2}",
    "critics called it {adj}, and I found it genuinely {adj2}",
    "the characters were {adj}, making for a {adj2} watch",
]

def generate_reviews(adjectives, feel_word, n):
    reviews = []
    for _ in range(n):
        template = random.choice(templates)
        reviews.append(template.format(
            adj=random.choice(adjectives),
            adj2=random.choice(adjectives),
            feel=feel_word,
        ))
    return reviews

generated_positive = generate_reviews(positive_adjectives, "loved", 60)
generated_negative = generate_reviews(negative_adjectives, "hated", 60)

texts = hand_written_positive + generated_positive + hand_written_negative + generated_negative
labels = ([1] * (len(hand_written_positive) + len(generated_positive)) +
          [0] * (len(hand_written_negative) + len(generated_negative)))

df = pd.DataFrame({"text": texts, "label": labels}).drop_duplicates(subset="text").reset_index(drop=True)

# ── Inject a small amount of label noise (real-world labels are never perfect) ─
# Without this, this toy dataset is a little TOO easy and every model below
# would hit 100% accuracy — which would teach you nothing about how to READ
# a confusion matrix or ROC curve. A few flipped labels makes it realistic.
noise_rng = np.random.RandomState(1)
n_noisy = int(0.06 * len(df))                       # flip ~6% of labels
noisy_indices = noise_rng.choice(df.index, size=n_noisy, replace=False)
df.loc[noisy_indices, "label"] = 1 - df.loc[noisy_indices, "label"]

print(f"Total examples: {len(df)}  |  Positive: {df['label'].sum()}  |  Negative: {(df['label']==0).sum()}")
df.sample(5, random_state=RANDOM_STATE)


In [ ]:
# ── Train/test split ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"],
    test_size=0.25,          # hold out 25% purely for final evaluation
    random_state=RANDOM_STATE,
    stratify=df["label"],     # ⚠️ IMPORTANT: keeps the same class balance in
                               # both splits — without this, a small/imbalanced
                               # dataset can accidentally put almost all of one
                               # class into just the train OR just the test set
)
print(f"Train size: {len(X_train_text)}  |  Test size: {len(X_test_text)}")


In [ ]:
# ── Vectorize with TF-IDF (Module 3) — fit on train, transform on test only ──
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)   # LEARN vocabulary + weights from train
X_test = vectorizer.transform(X_test_text)          # REUSE that vocabulary on test — never re-fit

print("Feature matrix shape (train):", X_train.shape)
print("Feature matrix shape (test): ", X_test.shape)


## Part 2 — Naive Bayes: the Classic Text Classifier

Naive Bayes answers: *"given these words, which class is most probable?"* using **Bayes' theorem**, with one deliberately simplifying ("naive") assumption — it treats every word's presence as **independent** of every other word, given the class. That assumption is technically wrong (word order and correlation obviously matter), but it works surprisingly well on text anyway, and it's cheap enough that it's still the textbook first pass for spam filtering.

`MultinomialNB` specifically expects **word-count-style** features, which is exactly what TF-IDF/BoW vectors are — this is why Naive Bayes and text vectorization are almost always taught together.


In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)              # "fit" = learn word-probability statistics per class

nb_predictions = nb_model.predict(X_test)    # hard class predictions (0 or 1)
nb_probabilities = nb_model.predict_proba(X_test)   # confidence scores per class

print("Predictions:  ", nb_predictions)
print("True labels:  ", y_test.values)
print("First 3 probability rows [P(negative), P(positive)]:")
print(nb_probabilities[:3].round(3))


In [ ]:
# 🔀 When to reach for Naive Bayes specifically
# | Situation                                   | Naive Bayes a good fit? |
# |-----------------------------------------------|----------------------------|
# | Need something trained in milliseconds           | Yes — trains almost instantly, even on huge data |
# | Very high-dimensional sparse features (BoW/TF-IDF) | Yes — this is its home turf |
# | Need well-calibrated probabilities                  | No — probabilities tend to be overconfident (see Part 11) |
# | Features are highly correlated with each other        | Weaker — violates the independence assumption more severely |
#
# 📋 COPY-PASTE TEMPLATE
# from sklearn.naive_bayes import MultinomialNB
# model = MultinomialNB(alpha=1.0)   # alpha = Laplace smoothing; higher alpha =
#                                     # more conservative about rare/unseen words
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)

print("Naive Bayes guidance shown above.")


## Part 3 — Logistic Regression: the Industry Default Baseline

If you had to pick **one** classical model to start any text classification project with, it's this one. Logistic Regression learns a **weight per feature** (per word/n-gram), sums them up, and squashes the result into a 0–1 probability. It's fast, scales to millions of features, produces genuinely usable probabilities (Part 11), and — critically for production — its weights are directly **interpretable**: you can literally read off which words push a prediction toward which class.


In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_model = LogisticRegression(
    max_iter=1000,       # text problems often need more iterations to converge
                          # than LogisticRegression's default of 100
    C=1.0,                # inverse regularization strength — smaller C = stronger
                           # regularization = simpler model, less prone to overfitting
    random_state=RANDOM_STATE,
)
logreg_model.fit(X_train, y_train)

logreg_predictions = logreg_model.predict(X_test)
print("Accuracy on test set:", (logreg_predictions == y_test.values).mean())


In [ ]:
# ── Interpretability: which words drove the predictions? ────────────────────
# This is the single biggest practical advantage Logistic Regression has over
# a black-box model — you can explain a prediction to a stakeholder in one
# sentence: "this review was classified positive mainly because of the words
# 'fantastic', 'brilliant', and 'wonderful'."

feature_names = vectorizer.get_feature_names_out()
coefficients = logreg_model.coef_[0]        # one weight per feature; higher = pushes toward class 1

top_positive_idx = np.argsort(coefficients)[-10:][::-1]   # 10 largest positive weights
top_negative_idx = np.argsort(coefficients)[:10]           # 10 largest negative weights

print("Top words pushing toward POSITIVE:")
for i in top_positive_idx:
    print(f"  {feature_names[i]:20s}  weight={coefficients[i]:.3f}")

print("\nTop words pushing toward NEGATIVE:")
for i in top_negative_idx:
    print(f"  {feature_names[i]:20s}  weight={coefficients[i]:.3f}")


In [ ]:
# 📋 COPY-PASTE TEMPLATE — the config you'll reuse most often
#
# model = LogisticRegression(
#     max_iter=1000,
#     C=1.0,
#     class_weight="balanced",   # ⚠️ set this whenever your classes are imbalanced
#                                 # (see Part 8) — free, one-line fix, do it by default
#     random_state=42,
# )
#
# 🔀 L1 vs L2 regularization (the `penalty` parameter)
# | penalty | Effect                                                          |
# |----------|--------------------------------------------------------------------|
# | "l2" (default) | Shrinks all weights smoothly toward zero — the safe default    |
# | "l1"            | Pushes many weights to EXACTLY zero — gives automatic feature  |
# |                  | selection, useful when you want a sparse, more interpretable model|
# (l1 requires solver="liblinear" or solver="saga")

print("Logistic Regression templates and regularization guidance shown above.")


## Part 4 — Linear Support Vector Machines

SVMs find the decision boundary that maximizes the **margin** (the gap) between classes, rather than just fitting probabilities. For text, `LinearSVC` (the linear-kernel SVM) is the one you'll actually use — TF-IDF features are already extremely high-dimensional and (roughly) linearly separable, so a non-linear kernel rarely helps and is far slower. `LinearSVC` and Logistic Regression are the two most common head-to-head baselines run against each other on any new text classification task.


In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=1.0,                  # same role as in Logistic Regression: smaller = more regularization
    random_state=RANDOM_STATE,
    max_iter=5000,
)
svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_test)
print("Accuracy on test set:", (svm_predictions == y_test.values).mean())

# ⚠️ LinearSVC has NO .predict_proba() by default — it outputs a raw decision
# score, not a calibrated probability. If you need probabilities (e.g. to set
# a confidence threshold, Part 11), either:
#   (a) use CalibratedClassifierCV(svm_model) to add calibrated probabilities, or
#   (b) just use LogisticRegression instead, which gives you probabilities natively


In [ ]:
# 🔀 LinearSVC vs SVC(kernel="rbf") — don't reach for the wrong one
# | Situation                                          | Choose             |
# |--------------------------------------------------------|----------------------|
# | Text data (high-dim, sparse, roughly linear)              | LinearSVC — much faster, scales to huge feature counts|
# | Small dataset, suspect a genuinely non-linear boundary      | SVC(kernel="rbf") — but rarely needed for text |
#
# 📋 COPY-PASTE TEMPLATE
# from sklearn.svm import LinearSVC
# from sklearn.calibration import CalibratedClassifierCV
#
# base_svm = LinearSVC(C=1.0, class_weight="balanced")
# svm_with_probabilities = CalibratedClassifierCV(base_svm)   # wraps it so
#                                                              # .predict_proba() works
# svm_with_probabilities.fit(X_train, y_train)

print("SVM guidance shown above.")


## Part 5 — Tree Ensembles: Random Forest & Gradient Boosting

Tree-based models generally aren't the first choice for raw high-dimensional sparse TF-IDF text (linear models tend to win there) — but they become the standard choice the moment you **combine text-derived features with structured/tabular features**: e.g. `tfidf_score` + `user_account_age_days` + `num_previous_purchases` + `is_verified` all in one feature vector for a fraud/abuse model. This "text features + business features together" pattern is extremely common in real industry systems (recommendation, trust & safety, ranking).

We use `RandomForestClassifier` (sklearn, no extra install) for the live demo below — it trains many decision trees on random subsets of the data/features and averages their votes, which tends to be a more forgiving, harder-to-misconfigure starting point than boosting on a small dataset like ours. `XGBoost` and `LightGBM` (gradient **boosting**, where trees are added one at a time to correct the previous trees' mistakes) are the two libraries you'll actually reach for at production scale — shown below as templates.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Tree models generally want DENSE input and work well when you combine a
# handful of engineered numeric features with the (dense) text features —
# exactly the realistic pattern below.

# Simple engineered features anyone can compute from raw text in one line each:
review_lengths_train = X_train_text.str.split().apply(len).values.reshape(-1, 1)
review_lengths_test = X_test_text.str.split().apply(len).values.reshape(-1, 1)

exclamation_counts_train = X_train_text.str.count("!").values.reshape(-1, 1)
exclamation_counts_test = X_test_text.str.count("!").values.reshape(-1, 1)

# Stack engineered numeric features together with the TF-IDF matrix itself
# (converted to dense with .toarray() — fine at this small scale; at real
# production scale you'd use a sparse-aware model or reduce dimensionality
# first, e.g. with the embeddings from Module 4 instead of raw TF-IDF):
X_train_combined = np.hstack([review_lengths_train, exclamation_counts_train, X_train.toarray()])
X_test_combined = np.hstack([review_lengths_test, exclamation_counts_test, X_test.toarray()])

rf_model = RandomForestClassifier(
    n_estimators=200,     # number of trees — more trees = more stable, diminishing returns past ~200-500
    random_state=RANDOM_STATE,
)
rf_model.fit(X_train_combined, y_train)
rf_predictions = rf_model.predict(X_test_combined)
print("Accuracy on test set:", (rf_predictions == y_test.values).mean())

# In a real project, the engineered features here would sit alongside genuine
# BUSINESS signals (account age, past behavior, verification status, etc.) —
# the POINT is the PATTERN: numeric business features + text-derived features,
# fed together into one tree model, is exactly how this shows up in production.


In [ ]:
# 📋 COPY-PASTE TEMPLATE — XGBoost (the most common production gradient-boosting choice)
#
# %pip install xgboost
# import xgboost as xgb
#
# model = xgb.XGBClassifier(
#     n_estimators=300,       # number of boosting rounds (trees)
#     max_depth=6,              # deeper = more expressive, more overfitting risk
#     learning_rate=0.05,        # smaller = needs more trees, but generalizes better
#     subsample=0.8,               # randomly sample 80% of rows per tree — reduces overfitting
#     colsample_bytree=0.8,         # randomly sample 80% of features per tree — same reason
#     eval_metric="logloss",
#     random_state=42,
# )
# model.fit(
#     X_train_combined, y_train,
#     eval_set=[(X_test_combined, y_test)],
#     verbose=False,
# )

# 🔀 Random Forest vs Gradient Boosting (XGBoost/LightGBM/HistGradientBoostingClassifier)
# | Situation                                              | Choose               |
# |------------------------------------------------------------|-----------------------|
# | Want something fast to configure, resistant to overfitting    | Random Forest          |
# | Have enough data to tune carefully, want the best possible score| Gradient boosting     |
# | Very large dataset (100k+ rows)                                  | LightGBM (fastest, lowest memory)|
# | Best all-around tooling/ecosystem support                          | XGBoost                |

print("Gradient boosting production template and comparison guidance shown above.")


## Part 6 — Comparing Models Fairly, Side by Side

Never trust a single accuracy number from a single split — that's the #1 beginner mistake. Compare candidate models on the **same** split using **multiple** metrics at once.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models_to_compare = {
    "Naive Bayes": nb_predictions,
    "Logistic Regression": logreg_predictions,
    "Linear SVM": svm_predictions,
    "Random Forest": rf_predictions,
}

comparison_rows = []
for name, preds in models_to_compare.items():
    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1": f1_score(y_test, preds),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("model").round(3)
comparison_df


## Part 7 — Evaluation, Properly

Accuracy alone lies to you the moment classes are imbalanced (a model that always predicts "not spam" on a dataset that's 99% not-spam gets 99% accuracy while being completely useless). This is the toolkit real teams actually use to judge a classifier.

- **Precision**: of everything the model called positive, how much was actually positive? (Cost of false positives.)
- **Recall**: of everything actually positive, how much did the model catch? (Cost of false negatives.)
- **F1**: the harmonic mean of precision and recall — one number balancing both.
- **Confusion matrix**: the full breakdown — exactly which mistakes are being made, not just how many.
- **ROC-AUC**: how well the model ranks positives above negatives across ALL thresholds, not just the default 0.5 one.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(y_test, logreg_predictions, target_names=["negative", "positive"]))


In [ ]:
cm = confusion_matrix(y_test, logreg_predictions)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative", "positive"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()

# Reading it: rows = TRUE label, columns = PREDICTED label.
# Top-left  = true negatives  (correctly predicted negative)
# Top-right = false positives (predicted positive, actually negative)
# Bottom-left  = false negatives (predicted negative, actually positive)
# Bottom-right = true positives (correctly predicted positive)


In [ ]:
# ── ROC curve and AUC ────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve

logreg_probabilities = logreg_model.predict_proba(X_test)[:, 1]   # probability of class 1
fpr, tpr, thresholds = roc_curve(y_test, logreg_probabilities)
auc_score = roc_auc_score(y_test, logreg_probabilities)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.3f}  (1.0 = perfect ranking, 0.5 = no better than random)")

# 🔀 ROC-AUC vs Precision-Recall AUC — pick the right one
# | Situation                                    | Prefer            |
# |--------------------------------------------------|---------------------|
# | Roughly balanced classes                            | ROC-AUC              |
# | Highly imbalanced classes (fraud, rare disease, abuse)| PR-AUC — ROC-AUC can look
# |                                                        | deceptively good on rare-positive problems
# precision, recall, pr_thresholds = precision_recall_curve(y_test, logreg_probabilities)


In [ ]:
# ── Cross-validation: don't trust a single train/test split at all ──────────
# A single split can be lucky/unlucky, especially on smaller datasets like
# ours. K-fold cross-validation trains/evaluates K times on different splits
# and reports the SPREAD of results, not just one number.

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# Wrap vectorizer + model in a Pipeline so each fold refits TF-IDF correctly
# on ONLY that fold's training portion — this avoids a subtle but common
# form of data leakage (fitting the vectorizer on data that includes what
# will be "held out" for evaluation in that fold).
cv_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv_scores = cross_val_score(cv_pipeline, df["text"], df["label"], cv=5, scoring="f1")
print("F1 scores across 5 folds:", cv_scores.round(3))
print(f"Mean F1: {cv_scores.mean():.3f}  (+/- {cv_scores.std():.3f})")


## Part 8 — Handling Class Imbalance

Real production text data is rarely 50/50 — fraud, toxic content, and rare-intent detection are often 99:1 or worse. Three practical fixes, in order of how often they're actually used:


In [ ]:
# ── Fix 1: class_weight="balanced" (free, always try this first) ────────────
# Tells the model to penalize mistakes on the MINORITY class more heavily
# during training, roughly proportional to how rare that class is —
# a one-line change, no extra data manipulation needed.

balanced_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
balanced_model.fit(X_train, y_train)
print("Trained with class_weight='balanced' — one-line fix, try this first on any imbalanced dataset.")


In [ ]:
# ── Fix 2: resampling (oversample minority / undersample majority) ──────────
# %pip install imbalanced-learn
#
# from imblearn.over_sampling import SMOTE
# # SMOTE = Synthetic Minority Oversampling TEchnique — generates synthetic
# # minority-class examples by interpolating between real ones, rather than
# # just duplicating existing rows (which would cause more overfitting).
#
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
# # fit your model on X_train_resampled / y_train_resampled instead of the original

print("SMOTE resampling pattern shown above — reach for this when class_weight "
      "alone isn't enough, typically on severe imbalance (>95:5).")


In [ ]:
# ── Fix 3: threshold tuning (adjust the decision boundary, not the data) ────
# By default, predict() calls something positive if P(positive) > 0.5. On
# imbalanced data, that default threshold is often wrong for your actual
# cost trade-off (e.g. missing a fraud case may be far worse than a false alarm).

custom_threshold = 0.3   # lower threshold -> catches MORE positives, at the
                          # cost of MORE false positives; tune this against
                          # your specific precision/recall trade-off
adjusted_predictions = (logreg_probabilities >= custom_threshold).astype(int)

print("Default threshold (0.5) recall:", recall_score(y_test, logreg_predictions))
print(f"Adjusted threshold ({custom_threshold}) recall:", recall_score(y_test, adjusted_predictions))

# 📋 COPY-PASTE TEMPLATE for choosing a threshold systematically:
# precisions, recalls, thresholds = precision_recall_curve(y_test, probabilities)
# Pick the threshold that hits your MINIMUM acceptable recall or precision —
# don't just eyeball it; write down the actual business requirement first.


## Part 9 — Hyperparameter Tuning

Don't guess `C=1.0`. Search over a grid of candidate values, using cross-validation to score each one, and let the data decide.


In [ ]:
from sklearn.model_selection import GridSearchCV

# 📋 COPY-PASTE TEMPLATE — this exact structure works for almost any sklearn model
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],           # note the "classifier__" prefix —
    "classifier__penalty": ["l2"],                  # required because "classifier" is
                                                     # the step name inside our Pipeline
}

grid_search = GridSearchCV(
    cv_pipeline,             # the Pipeline defined in Part 7 (vectorizer + model together)
    param_grid,
    cv=5,                     # 5-fold cross-validation for each candidate combination
    scoring="f1",
    n_jobs=-1,                 # use all available CPU cores to search in parallel
)
grid_search.fit(df["text"], df["label"])

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated F1:", round(grid_search.best_score_, 3))
best_model = grid_search.best_estimator_   # the final, fully-fitted Pipeline — ready to use


In [ ]:
# 🔀 GridSearchCV vs RandomizedSearchCV vs Optuna
# | Tool                  | Use when...                                           |
# |--------------------------|-----------------------------------------------------|
# | GridSearchCV               | Small number of hyperparameters, few values each      |
# | RandomizedSearchCV          | Larger search space — samples randomly instead of     |
# |                              | trying every combination, much cheaper                 |
# | Optuna / Ray Tune              | Production-grade searches — smarter, adaptive search   |
# |                                 | strategies (Bayesian optimization), used for the        |
# |                                 | biggest, most expensive tuning jobs                     |

print("Tuning tool comparison shown above.")


## Part 10 — Interpretability & Explainability

Part 3 already showed the simplest form of this (reading Logistic Regression's coefficients directly). For more complex models — trees, or when you need per-*prediction* explanations rather than global feature importance — reach for **SHAP**, the industry-standard explainability library.


In [ ]:
# %pip install shap
# import shap
#
# explainer = shap.Explainer(rf_model, X_train_combined)
# shap_values = explainer(X_test_combined)
#
# feature_names = ["review_length", "exclamation_count"] + list(vectorizer.get_feature_names_out())
# shap.summary_plot(shap_values, X_test_combined, feature_names=feature_names)
# # Shows which features matter most OVERALL, and in which direction, across
# # every prediction — the standard plot data scientists put in a model report.
#
# shap.plots.waterfall(shap_values[0])
# # Explains ONE SPECIFIC prediction: exactly how much each feature pushed
# # that particular example's score up or down from the baseline — this is
# # what you'd show a stakeholder asking "why did the model flag THIS case?"

print("SHAP usage pattern shown above — the standard tool for explaining tree-model "
      "predictions in production (also works on linear models and neural networks).")


## Part 11 — From Notebook Model to Production System

A model that scores well in a notebook is not automatically safe to deploy. Three things every production ML team checks before shipping a classifier:

### 1. Calibration — do the probabilities MEAN anything?
A model can have great accuracy while outputting garbage probabilities (e.g. always predicting 0.9 or 0.1, never anything in between). If any downstream logic uses the probability itself — not just the final 0/1 label — check calibration first.


In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_test, logreg_probabilities, n_bins=5)

plt.figure(figsize=(5, 5))
plt.plot(prob_pred, prob_true, marker="o", label="Logistic Regression")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of actual positives")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()

# Reading it: if the model's line sits ON the diagonal, when it says "80%
# confident," it's actually right about 80% of the time — trustworthy.
# Points well below the diagonal mean the model is OVERconfident.


### 2. Monitoring for drift
The world changes after you deploy — new slang, new topics, new user behavior — and your model's accuracy silently degrades as the live data drifts away from what it was trained on. In production this is tracked by continuously logging prediction distributions and (where labels eventually arrive, e.g. via user feedback) real accuracy over time, with alerts if either shifts sharply.

### 3. A/B testing before a full rollout
Never ship a new model to 100% of traffic at once. Standard practice: route a small percentage of real traffic to the new model, compare its live business metrics against the current production model, and only ramp up once it's proven itself on real, not just held-out, data.


In [ ]:
# ── 📋 COPY-PASTE TEMPLATE: the full production pipeline, saved and ready ────

import joblib

# 1. Build the final pipeline with your best hyperparameters found in Part 9
final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, C=1.0,
                                        class_weight="balanced", random_state=RANDOM_STATE)),
])
final_pipeline.fit(df["text"], df["label"])   # fit on ALL available labeled data for the final model

# 2. Save it as a single artifact
# joblib.dump(final_pipeline, "sentiment_classifier_v1.joblib")

# 3. Later, in a completely separate production process:
# loaded_pipeline = joblib.load("sentiment_classifier_v1.joblib")
# probabilities = loaded_pipeline.predict_proba(["a brand new review to classify"])[:, 1]
# prediction = "positive" if probabilities[0] >= 0.5 else "negative"

print("Final production pipeline built and fitted. Save/load pattern shown above.")


## Recap & What's Next

You can now train, evaluate, tune, interpret, and productionize the exact family of models that quietly power most of the text classification happening inside big tech companies today — not despite LLMs existing, but *because of* the latency/cost gap between the two. You have working templates for Naive Bayes, Logistic Regression, LinearSVC, gradient-boosted trees, cross-validation, `GridSearchCV`, imbalance handling, calibration checks, and a save/load production pipeline.

### Try this before the next lesson
1. Swap in your own labeled dataset from Module 1 and rerun Parts 2–4 — compare which classical model wins on YOUR data (it varies by dataset).
2. Run `GridSearchCV` over both `C` and `ngram_range` together and see how much it improves cross-validated F1 versus the defaults.
3. Plot a calibration curve for your best model before you'd trust its probabilities for anything threshold-based.

### Next lesson in your NLP mastery path
**Module 6: Neural Networks for NLP — RNNs, LSTMs, and the Road to Transformers** — why sequence order matters enough to need a fundamentally different architecture, how recurrent networks process text step by step, where they fall short, and the exact gap that Transformers (and models like Claude) were built to close.
